In [ ]:
import pandas as pd
import numpy as np
import nltk
import math
from collections import Counter
from nltk.stem import WordNetLemmatizer

In [ ]:
df = pd.read_csv("brown.csv")

In [ ]:
df.head()

,filename,para_id,sent_id,raw_text,tokenized_text,tokenized_pos,label
0,cd05,0,0,"Furthermore/rb ,/, as/cs an/at encouragement/n...","Furthermore , as an encouragement to revisioni...","rb , cs at nn in nn nn , pps rb bez jj to vb c...",religion
1,cd05,0,1,The/at Unitarian/jj clergy/nns were/bed an/at ...,The Unitarian clergy were an exclusive club of...,at jj nns bed at jj nn in vbn nns -- cs at nn ...,religion
2,cd05,0,2,"Ezra/np Stiles/np Gannett/np ,/, an/at honorab...","Ezra Stiles Gannett , an honorable representat...","np np np , at jj nn in at nn , vbd ppl rb in a...",religion
3,cd05,0,3,"Even/rb so/rb ,/, Gannett/np judiciously/rb ar...","Even so , Gannett judiciously argued , the Ass...","rb rb , np rb vbd , at nn-tl md rb vb cs np ``...",religion
4,cd05,0,4,We/ppss today/nr are/ber not/* entitled/vbn to...,We today are not entitled to excoriate honest ...,ppss nr ber * vbn to vb jj nns wps vbd np to b...,religion


In [ ]:
from nltk.corpus import brown
nltk.download('brown')




[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Unzipping corpora/brown.zip.


True

In [ ]:
word = [w.lower() for w in brown.words()]

In [ ]:
split = int(len(word)*0.8)
train = word[:split]
test = word[split:]

In [ ]:
unigram = Counter(train)
total_word = len(train)
vocab = len(unigram)

In [ ]:
bigrams = Counter(zip(train[:-1], train[1:]))
unigram_context = Counter(train[:-1])


In [ ]:
trigrams = Counter(zip(train[:-2], train[1:-1], train[2:]))
unigram_context2 = Counter(zip(train[:-2], train[-1:1]))

In [ ]:
def unigram_prob(word):
    return unigram[word]/total_word

In [ ]:
def bigram_prob(word1, word2):
    if (word1, word2) in bigrams:
        return bigrams[(word1, word2)]/unigram_context[word1]
    else:
        return 0

In [ ]:
def trigram_prob(word1, word2, word3):
    if (word1, word2, word3) in trigrams:
        return trigrams[(word1, word2, word3)]/unigram_context2[(word1, word2)]

In [ ]:
def unigram_prob_laplace(word):
    return (unigram[word]+1)/(total_word+vocab)

In [ ]:
def bigram_prob_laplace(word1, word2):

    if (word1, word2) in bigrams:
        return (bigrams[(word1, word2)] + 1) / (unigram_context[word1] + vocab)
    else:

        return 1 / (unigram_context[word1] + vocab) if word1 in unigram_context else 1 / vocab

In [ ]:
def trigram_prob_laplace(word1, word2, word3):
  return((trigrams[(word1, word2, word3)]+1)/(unigram_context2[(word1, word2)]+vocab))

In [ ]:
def perplexity_unigram(test):
  log_prob =0
  for word in test:
     p = unigram_prob_laplace(word)
     log_prob += math.log(p)
  return math.exp(-log_prob/len(test))

In [ ]:

def perplexity_bigram(test):
  log_prob =0
  for i in range(len(test)-1):
     p = bigram_prob_laplace(test[i], test[i+1])
     log_prob += math.log(p)
  return math.exp(-log_prob/(len(test)-1))


In [ ]:
def perplexity_trigram(test):
  log_prob =0
  for i in range(len(test)-2):
     p = trigram_prob_laplace(test[i], test[i+1], test[i+2])
     log_prob += math.log(p)
  return math.exp(-log_prob/(len(test)-2))

In [ ]:
def unigram_perplexity(sentence):
    words = sentence.lower().split()

    log_prob = 0

    for word in words:
        p = unigram_prob_laplace(word)
        log_prob += math.log(p)

    return math.exp(-log_prob / len(words))

def bigram_perplexity(sentence):
    words = sentence.lower().split()

    if len(words) < 2:
        return float('inf')

    log_prob = 0

    for i in range(1, len(words)):
        p = bigram_prob_laplace(words[i-1], words[i])
        log_prob += math.log(p)

    return math.exp(-log_prob / (len(words)-1))

def trigram_perplexity(sentence):
    words = sentence.lower().split()

    if len(words) < 3:
        return float('inf')

    log_prob = 0

    for i in range(2, len(words)):
        p = trigram_prob_laplace(words[i-2], words[i-1], words[i]) # Use Laplace smoothed probability
        log_prob += math.log(p)

    return math.exp(-log_prob / (len(words)-2))
sentence = input("Enter a sentence: ")

print("\nSentence:", sentence)

print("Unigram Perplexity :", unigram_perplexity(sentence))
print("Bigram Perplexity  :", bigram_perplexity(sentence))
print("Trigram Perplexity :", trigram_perplexity(sentence))

Enter a sentence: sameer is my friend

Sentence: sameer is my friend
Unigram Perplexity : 6070.440605118734
Bigram Perplexity  : 14288.53137791531
Trigram Perplexity : 44797.000000000015


### Calculating Perplexity on the Test Set

Now, let's calculate the perplexity for the entire `test` dataset using our Laplace-smoothed probability functions. This gives us a more general measure of how well our models predict unseen sequences of words.

In [ ]:
print("Calculating perplexity for the test set...")

unigram_test_perplexity = perplexity_unigram(test)
bigram_test_perplexity = perplexity_bigram(test)
trigram_test_perplexity = perplexity_trigram(test)

print(f"\nUnigram Perplexity on Test Set: {unigram_test_perplexity:.2f}")
print(f"Bigram Perplexity on Test Set: {bigram_test_perplexity:.2f}")
print(f"Trigram Perplexity on Test Set: {trigram_test_perplexity:.2f}")

Calculating perplexity for the test set...

Unigram Perplexity on Test Set: 1110.45
Bigram Perplexity on Test Set: 4938.65
Trigram Perplexity on Test Set: 26783.39
